# Decorador AR para interiores

Este cuaderno sigue el recorrido desde la detección de pared y suelo hasta una habitación decorada con cuadros, alfombras y muebles 3D. Ejecuta las celdas en orden con el kernel de `.venv`.

La primera ejecución prepara las vistas de [ZInD](https://github.com/zillow/zind/tree/main/sample_tour/000) y carga SegFormer. Cada ejecución guarda sus resultados en una carpeta nueva.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/decorador_ar').is_dir())
sys.path.insert(0, str(root / 'src'))

import matplotlib.pyplot as plt
from decorador_ar.photo import Segmenter
from decorador_ar.depth import evaluate_files
from decorador_ar.image_io import write_image
from decorador_ar.composition_editor import CompositionEditor
from decorador_ar.photo_furniture import MODELS
from decorador_ar import notebook_workflow as workflow

SCENE = 'bonus_02'
output = workflow.new_run(root)
segmenter = Segmenter()
print(f'Resultados en {output}')

## 1. Foto real, pared y suelo

Empezamos con seis vistas de una vivienda y añadimos otras seis en el bloque 3D. SegFormer identifica pared y suelo. Las aristas anotadas por ZInD dan la referencia geométrica para rectificar ambos planos, como en la práctica 1.

In [ ]:
scenes = workflow.load_scenes(root, segmenter)
print('Vistas disponibles:', ', '.join(scenes))
scene = scenes[SCENE]
workflow.show_original(scene, SCENE)
plt.show()

## 2. Rectificar y diseñar la habitación

La línea amarilla debe seguir el rodapié en la foto y quedar horizontal en el plano rectificado. El encaje comprueba dónde cae la pieza según la máscara. Debajo se muestran seis composiciones con tamaños y combinaciones elegidos para cada vista. El catálogo reúne tres cuadros enmarcados y tres alfombras del Met.

In [ ]:
workflow.show_rectification(scene)
plt.show()
workflow.show_gallery(scenes)
plt.show()

## 3. Evaluar la segmentación

ADE20K aporta máscaras anotadas para comparar pared y suelo mediante IoU. Con el ZIP local, se evalúan nueve fotos y se guarda el CSV. El estudio usa el colocador experimental de ADE20K y sus resultados de colocación se interpretan por separado de la rectificación de ZInD.

In [ ]:
archive = root / 'data/raw/ADEChallengeData2016.zip'
if archive.is_file():
    scores = workflow.evaluate_masks(archive, output / 'ade20k', segmenter)
    print('IoU media por superficie:', scores)
    print(output / 'ade20k/resumen.csv')
else:
    print('Evaluación ADE20K disponible al guardar el ZIP en data/raw/.')

## 4. Colocar muebles 3D en fotos

Proyectamos una mesa baja y un banco, con patas y tablero, usando la cámara y el plano del suelo de ZInD. La huella debe quedar sobre suelo detectado. En las seis vistas nuevas incluimos dos pasillos para comprobar por qué el encaje de la base no basta para decidir si un mueble estorba. Las doce vistas vienen de una sola vivienda.

Para probar otra posición, cambia el centro en píxeles, el modelo, la escala o el giro. El render ordena las superficies del propio mueble por profundidad. La ocultación detrás de objetos reales se incorporará al medir la profundidad de la habitación.

In [ ]:
stock = workflow.load_furniture_stock(root, segmenter)
workflow.show_furniture_stock(stock)

FURNITURE_SCENE = 'bonus_02'
MODEL, SCALE, YAW = 'mesa_baja', .73, 0
ANCHOR = (416, 576)  # Centro de apoyo (x, y). Pon None para usar el punto de control.
print('Modelos:', ', '.join(MODELS))
furniture_scene = (scenes | stock)[FURNITURE_SCENE]
base = workflow.decorate_2d(FURNITURE_SCENE, furniture_scene) if FURNITURE_SCENE in scenes else None
try:
    preview = workflow.furniture_preview(furniture_scene, ANCHOR or furniture_scene.floor_anchor,
                                         model=MODEL, scale=SCALE, yaw=YAW, image=base)
except ValueError as error:
    print('Colocación rechazada:', error)
else:
    workflow.show_furniture_detail(furniture_scene, preview)
    plt.show()
    write_image(output / 'mueble-3d.png', preview.image)

## 5. Evaluar profundidad cuando haya mapas disponibles

La evaluación opcional compara dos mapas alineados y expresados en metros. Servirá para validar la estimación de profundidad con fotos RGB-D antes de usarla para ocultar muebles.

In [ ]:
DEPTH_REFERENCE = None  # Mapa .npy de referencia.
DEPTH_PREDICTION = None  # Predicción alineada.
if DEPTH_REFERENCE is not None and DEPTH_PREDICTION is not None:
    print(evaluate_files(DEPTH_REFERENCE, DEPTH_PREDICTION, output / 'profundidad.json', .1, 10))

## Componer una habitación

Elige una vista, una superficie y una pieza. La cruz marca el punto X, Y. Para un cuadro o una alfombra, indica su esquina superior derecha. Para un mueble, indica el centro de su base. Con **Previsualizar pieza**, puedes probar posición, escala y giro. Pulsa **Añadir en X, Y** para fijar la pieza.

Prueba `cocina_01` → **En altura** → **Detectar apoyos**. El contorno cian marca la parte utilizable de la encimera y las coordenadas se rellenan con su centro. Elige **Plato**, revisa la vista previa y confirma. La primera detección descarga los pesos de profundidad. La altura se estima a partir de la foto y algunas superficies estrechas o tapadas se descartan.

Los tableros y baldas de muebles virtuales aparecen en **Apoyo** al colocarlos. Si una superficie de la foto no se detecta, pulsa **Marcar tablero**, indica su altura relativa a la cámara y registra las cuatro esquinas en orden con **Marcar X, Y**. El editor comprueba que la base cabe en el apoyo y respeta el espacio entre baldas virtuales.

Activa o desactiva **Luz y sombra** para comparar cómo se integran las piezas 3D. **Guardar PNG** exporta las piezas confirmadas con el ajuste de luz actual.

In [ ]:
CLICKABLE = False  # Actívalo si el visor de Jupyter admite clics sobre la imagen.
if CLICKABLE:
    get_ipython().run_line_magic('matplotlib', 'widget')
support_scenes = workflow.load_support_scenes(root, segmenter)
editor = CompositionEditor(scenes | stock | support_scenes, SCENE, output).show(clickable=CLICKABLE)

In [ ]:
print(f'Las composiciones exportadas se guardan en {output}')